# 3. Probability Distributions

**Statistical Foundations for Data Science — Notebook 3 of 8**

A **distribution** is a reusable template for randomness. Instead of describing every
random variable from scratch, statisticians noticed that the same handful of shapes keep
appearing — the number of clicks on an ad, the number of defects on a production line,
the height of a person, the time until a server fails. Learn the templates and you can
model most of the world with two or three parameters.

### What you will learn

**Discrete:** Bernoulli · Binomial · Poisson · Geometric
**Continuous:** Uniform · Normal · Exponential · Log-normal · t · Chi-square · F

Plus:
- The **empirical rule** (68–95–99.7) and z-scores
- How to **choose** a distribution for a real problem
- How to **fit** a distribution to data and check the fit (Q–Q plots, goodness-of-fit)
- The **Central Limit Theorem** preview (developed fully in Notebook 4)

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy import stats

rng = np.random.default_rng(seed=2024)
plt.rcParams["figure.figsize"] = (8, 4)
plt.rcParams["axes.grid"] = True

def plot_pmf(dist, k_max, title, ax=None):
    '''Draw the PMF of a discrete scipy distribution.'''
    ax = ax or plt.gca()
    k = np.arange(0, k_max + 1)
    ax.bar(k, dist.pmf(k), color="steelblue", edgecolor="black", alpha=0.85)
    ax.set_title(title, fontsize=10)
    ax.set_xlabel("k"); ax.set_ylabel("P(X = k)")
    return ax

---
## Part A — Discrete distributions

## 3.1 Bernoulli: a single yes/no trial

$$X \sim \text{Bernoulli}(p), \qquad P(X=1) = p,\ P(X=0) = 1-p$$

$$E[X] = p, \qquad \operatorname{Var}(X) = p(1-p)$$

**Use it for:** one click / no click, one conversion, one defective item, one correct
answer. It is the atom that every other discrete distribution is built from.

Notice that the variance $p(1-p)$ is maximised at $p = 0.5$ — a fair coin is the most
unpredictable one.

In [ ]:
ps = np.linspace(0, 1, 201)
plt.plot(ps, ps * (1 - ps), lw=2, color="steelblue")
plt.axvline(0.5, ls="--", color="crimson")
plt.xlabel("p"); plt.ylabel("Var(X) = p(1-p)")
plt.title("A Bernoulli trial is most uncertain when p = 0.5")
plt.show()

b = stats.bernoulli(p=0.3)
print(f"P(X=1) = {b.pmf(1):.2f}, P(X=0) = {b.pmf(0):.2f}")
print(f"mean = {b.mean():.2f}, var = {b.var():.2f}")
print("10 draws:", b.rvs(10, random_state=0))

---
## 3.2 Binomial: counting successes in $n$ trials

Add up $n$ **independent** Bernoulli trials with the same $p$:

$$X \sim \text{Bin}(n, p), \qquad P(X = k) = \binom{n}{k} p^k (1-p)^{n-k}$$

$$E[X] = np, \qquad \operatorname{Var}(X) = np(1-p)$$

**The four conditions (remember them as BINS):**
- **B**inary outcomes
- **I**ndependent trials
- **N**umber of trials fixed in advance
- **S**ame probability $p$ every trial

**Use it for:** conversions out of 1,000 visitors, defective items in a batch of 50,
correct answers on a 20-question quiz, heads in 10 flips.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, (n, p) in zip(axes, [(10, 0.2), (10, 0.5), (50, 0.5)]):
    plot_pmf(stats.binom(n, p), n, f"Binomial(n={n}, p={p})\nmean={n*p:.1f}, var={n*p*(1-p):.2f}", ax)
plt.tight_layout(); plt.show()

print("Small p -> right-skewed. p = 0.5 -> symmetric. Large n -> approaches a bell curve.")

In [ ]:
# Worked example: an email campaign is sent to 500 people with a 4% click rate.
n, p = 500, 0.04
X = stats.binom(n, p)

print(f"Expected clicks        : {X.mean():.1f}  (sd {X.std():.2f})")
print(f"P(exactly 20 clicks)   : {X.pmf(20):.4f}")
print(f"P(at most 15 clicks)   : {X.cdf(15):.4f}")
print(f"P(more than 30 clicks) : {X.sf(30):.4f}")
print(f"Middle 95% of outcomes : {X.ppf(0.025):.0f} to {X.ppf(0.975):.0f} clicks")

In [ ]:
# Verify the formula against brute-force simulation
sim = rng.binomial(n, p, size=200_000)
k = np.arange(5, 40)

plt.hist(sim, bins=np.arange(-0.5, 45, 1), density=True, alpha=0.55,
         color="steelblue", label="simulation (200k campaigns)")
plt.plot(k, X.pmf(k), "o-", color="crimson", label="Binomial PMF")
plt.xlabel("number of clicks"); plt.ylabel("probability")
plt.title("Theory matches simulation")
plt.legend(); plt.show()

---
## 3.3 Poisson: counting events in a fixed window

When events happen at a constant average rate $\lambda$, independently, the count in a
fixed interval follows:

$$X \sim \text{Poisson}(\lambda), \qquad P(X = k) = \frac{\lambda^k e^{-\lambda}}{k!}$$

$$E[X] = \lambda, \qquad \operatorname{Var}(X) = \lambda$$

The mean equals the variance — a fingerprint you can check in your data. If the variance
is much larger, your data is **overdispersed** (use a Negative Binomial instead).

**Use it for:** support tickets per hour, website visits per minute, typos per page,
earthquakes per year, goals per football match.

**Relationship to Binomial:** Poisson is the limit of $\text{Bin}(n, p)$ as
$n \to \infty$, $p \to 0$ with $np = \lambda$ fixed. So "many trials, tiny probability
each" → Poisson.

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(14, 3.6))
for ax, lam in zip(axes, [1, 4, 15]):
    plot_pmf(stats.poisson(lam), int(lam * 3 + 8), f"Poisson(lambda={lam})", ax)
plt.tight_layout(); plt.show()

# Binomial -> Poisson convergence
lam = 3
print(f"{'n':>8} {'p':>8}   P(X=2) Binomial   P(X=2) Poisson")
for n in [10, 100, 1_000, 100_000]:
    p = lam / n
    print(f"{n:>8} {p:>8.5f}   {stats.binom(n, p).pmf(2):>15.6f}   {stats.poisson(lam).pmf(2):>14.6f}")

In [ ]:
# Worked example: a helpdesk receives on average 6 tickets per hour.
lam = 6
T = stats.poisson(lam)

print(f"P(no tickets in an hour)     : {T.pmf(0):.4f}")
print(f"P(exactly 6)                 : {T.pmf(6):.4f}")
print(f"P(more than 10)              : {T.sf(10):.4f}")
print(f"Staffing level covering 99%  : {T.ppf(0.99):.0f} tickets/hour")

# Scaling the window scales lambda: 15 minutes -> lambda = 1.5
print(f"\nP(no tickets in 15 minutes)  : {stats.poisson(lam/4).pmf(0):.4f}")

In [ ]:
# Diagnostic: is real count data actually Poisson? Compare mean and variance.
poisson_like = rng.poisson(6, 5_000)
overdispersed = rng.negative_binomial(2, 2/8, 5_000)   # same-ish mean, bigger variance

for name, d in [("Poisson-like counts", poisson_like), ("Overdispersed counts", overdispersed)]:
    print(f"{name:22s} mean={d.mean():6.2f}  var={d.var():6.2f}  "
          f"ratio={d.var()/d.mean():.2f}")
print("\nVariance/mean ratio near 1 supports Poisson; much greater than 1 does not.")

---
## 3.4 Geometric: waiting for the first success

$$X \sim \text{Geom}(p), \qquad P(X = k) = (1-p)^{k-1} p, \quad k = 1, 2, 3, \dots$$

$$E[X] = \frac{1}{p}, \qquad \operatorname{Var}(X) = \frac{1-p}{p^2}$$

**Use it for:** number of calls until a sale, retries until a request succeeds, ad
impressions until a click.

The geometric distribution is **memoryless**: after 10 failures, the expected additional
wait is *still* $1/p$. The coin has no memory of your bad luck — the gambler's fallacy in
one line of maths.

In [ ]:
p = 0.2
G = stats.geom(p)

plot_pmf(G, 30, f"Geometric(p={p}): trials until first success")
plt.show()

print(f"E[trials] = 1/p = {G.mean():.1f}")
print(f"P(success on first try)  = {G.pmf(1):.4f}")
print(f"P(needing more than 10)  = {G.sf(10):.4f}")

# Memorylessness: P(X > 15 | X > 5) == P(X > 10)
print(f"\nP(X > 15 | X > 5) = {G.sf(15)/G.sf(5):.4f}")
print(f"P(X > 10)         = {G.sf(10):.4f}   <- identical: the process has no memory")

---
## Part B — Continuous distributions

## 3.5 Uniform: every value equally likely

$$X \sim U(a, b), \qquad f(x) = \frac{1}{b-a} \text{ for } a \le x \le b$$

$$E[X] = \frac{a+b}{2}, \qquad \operatorname{Var}(X) = \frac{(b-a)^2}{12}$$

**Use it for:** modelling total ignorance within a range, random number generation, and
as the base for simulating *any* other distribution via the inverse-CDF trick.

In `scipy`, the parameters are `loc = a` and `scale = b - a`, not `a` and `b`. This trips
up everyone once.

In [ ]:
U = stats.uniform(loc=10, scale=20)          # Uniform(10, 30)
xs = np.linspace(5, 35, 500)

plt.plot(xs, U.pdf(xs), lw=2, color="steelblue")
plt.fill_between(xs, U.pdf(xs), alpha=0.3, color="steelblue")
plt.title("Uniform(10, 30)"); plt.xlabel("x"); plt.ylabel("f(x)")
plt.show()

print(f"mean = {U.mean():.2f}, var = {U.var():.2f}  (theory: (30-10)^2/12 = {(20**2)/12:.2f})")
print(f"P(15 < X < 22) = {U.cdf(22) - U.cdf(15):.4f}")

In [ ]:
# Inverse-CDF (inverse transform) sampling:
# feed uniform numbers into any distribution's ppf and you get samples from it.
u = rng.random(100_000)
exp_samples = stats.expon(scale=200).ppf(u)

plt.hist(exp_samples, bins=100, density=True, alpha=0.6, color="steelblue",
         label="samples from uniform + ppf")
xs = np.linspace(0, 1200, 400)
plt.plot(xs, stats.expon(scale=200).pdf(xs), lw=2, color="crimson", label="true Exponential PDF")
plt.title("Inverse transform sampling: uniforms in, any distribution out")
plt.legend(); plt.show()

---
## 3.6 Normal (Gaussian): the bell curve

$$X \sim N(\mu, \sigma^2), \qquad f(x) = \frac{1}{\sigma\sqrt{2\pi}} e^{-\frac{(x-\mu)^2}{2\sigma^2}}$$

The most important distribution in statistics, for one deep reason: the **Central Limit
Theorem** says that *sums and averages of many independent influences are approximately
Normal, whatever the original distribution*. Height, measurement error, and sample means
are all sums of many small effects.

**Properties:**
- Symmetric and bell-shaped; mean = median = mode
- Fully determined by $\mu$ (location) and $\sigma$ (spread)
- Any linear transform of a Normal is Normal
- Sum of independent Normals is Normal

**The empirical rule (68–95–99.7):** about 68% of values lie within $1\sigma$ of the mean,
95% within $2\sigma$, and 99.7% within $3\sigma$.

In [ ]:
xs = np.linspace(-6, 12, 600)
for mu, sd, style in [(0, 1, "-"), (3, 1, "--"), (3, 2.5, ":")]:
    plt.plot(xs, stats.norm(mu, sd).pdf(xs), style, lw=2, label=f"N(mu={mu}, sd={sd})")
plt.legend(); plt.title("mu shifts the curve, sigma stretches it")
plt.xlabel("x"); plt.ylabel("density")
plt.show()

In [ ]:
Z = stats.norm(0, 1)
print("The empirical rule, computed exactly:")
for k in (1, 2, 3):
    print(f"  P(|Z| < {k}) = {Z.cdf(k) - Z.cdf(-k):.6f}")

xs = np.linspace(-4, 4, 800)
plt.plot(xs, Z.pdf(xs), color="black", lw=1.5)
for k, colour, alpha in [(3, "steelblue", 0.20), (2, "steelblue", 0.30), (1, "steelblue", 0.45)]:
    m = (xs >= -k) & (xs <= k)
    plt.fill_between(xs[m], Z.pdf(xs[m]), color=colour, alpha=alpha)
for k, lbl in [(1, "68%"), (2, "95%"), (3, "99.7%")]:
    plt.text(k - 0.5, 0.05, lbl, fontsize=9)
plt.title("Empirical rule: 68 - 95 - 99.7")
plt.xlabel("z"); plt.ylabel("density")
plt.show()

In [ ]:
# Worked example: IQ scores are standardised to N(100, 15^2)
iq = stats.norm(100, 15)

print(f"P(IQ > 130)          = {iq.sf(130):.4f}  (about 1 in {1/iq.sf(130):.0f} people)")
print(f"P(85 < IQ < 115)     = {iq.cdf(115) - iq.cdf(85):.4f}")
print(f"99th percentile      = {iq.ppf(0.99):.1f}")
print(f"z-score for IQ 145   = {(145 - 100)/15:.2f}")

# Converting to and from z-scores
score = 128
z = (score - 100) / 15
print(f"\nIQ {score} -> z = {z:.2f} -> percentile = {stats.norm.cdf(z)*100:.1f}%")

---
## 3.7 Exponential: waiting time between events

The continuous cousin of the geometric distribution. If events arrive as a Poisson process
with rate $\lambda$, the **gap between consecutive events** is Exponential:

$$f(x) = \lambda e^{-\lambda x}, \quad x \ge 0, \qquad E[X] = \frac{1}{\lambda}, \qquad \operatorname{Var}(X) = \frac{1}{\lambda^2}$$

Note that mean = standard deviation. In `scipy`, use `scale = 1/lambda`.

**Use it for:** time until the next customer arrives, time until a component fails,
API latency, time between earthquakes.

Like the geometric, it is **memoryless**: a component that has run for 1,000 hours has the
same remaining expected life as a brand-new one. That is a strong assumption — real
machines wear out, which is why reliability engineers reach for the **Weibull**
distribution instead.

In [ ]:
lam = 1/200                       # mean 200 ms
E = stats.expon(scale=1/lam)
xs = np.linspace(0, 1000, 500)

plt.plot(xs, E.pdf(xs), lw=2, color="steelblue")
plt.axvline(E.mean(), color="crimson", ls="--", label=f"mean = {E.mean():.0f} ms")
plt.axvline(E.ppf(0.5), color="darkgreen", ls="--", label=f"median = {E.ppf(0.5):.0f} ms")
plt.axvline(E.ppf(0.95), color="orange", ls="--", label=f"p95 = {E.ppf(0.95):.0f} ms")
plt.title("Exponential response times: the mean is NOT the typical value")
plt.xlabel("milliseconds"); plt.ylabel("density"); plt.legend()
plt.show()

print(f"mean = {E.mean():.1f},  sd = {E.std():.1f}   (equal, as expected)")
print(f"P(X > 600) = {E.sf(600):.4f}")

In [ ]:
# The Poisson / Exponential duality, verified by simulation.
# Simulate arrival gaps from Exponential(mean 1/6 hour) and count arrivals per hour.
gaps = rng.exponential(scale=1/6, size=200_000)
arrival_times = np.cumsum(gaps)
counts_per_hour = np.bincount(arrival_times.astype(int))

print(f"Mean arrivals per hour  : {counts_per_hour.mean():.3f}   (Poisson lambda = 6)")
print(f"Variance                : {counts_per_hour.var():.3f}   (Poisson var = 6)")

---
## 3.8 Log-normal: when the *logarithm* is Normal

If $\log X \sim N(\mu, \sigma^2)$ then $X$ is log-normal. It arises whenever effects
**multiply** rather than add — and that is extremely common.

**Use it for:** income, house prices, city sizes, stock prices, file sizes, time spent on
a page. Always positive, always right-skewed.

**Practical tip:** if a feature is log-normal, take `np.log1p(x)` before feeding it to a
linear model. This single transformation fixes more regression problems than any other.

In [ ]:
income = rng.lognormal(mean=10.5, sigma=0.6, size=20_000)

fig, ax = plt.subplots(1, 2, figsize=(12, 4))
ax[0].hist(income, bins=80, color="steelblue", edgecolor="none")
ax[0].axvline(income.mean(),     color="crimson",   lw=2, label=f"mean   = {income.mean():,.0f}")
ax[0].axvline(np.median(income), color="darkgreen", lw=2, label=f"median = {np.median(income):,.0f}")
ax[0].set_title("Raw income: right-skewed"); ax[0].legend(fontsize=8)

ax[1].hist(np.log(income), bins=80, color="seagreen", edgecolor="none")
ax[1].set_title("log(income): approximately Normal")
plt.tight_layout(); plt.show()

print(f"Skewness raw       : {stats.skew(income):.2f}")
print(f"Skewness after log : {stats.skew(np.log(income)):.2f}")
print("\nThe mean is far above the median -> reporting 'average income' misleads.")

---
## 3.9 The three sampling distributions: t, Chi-square and F

These are not used to model raw data. They describe the behaviour of **statistics
computed from samples**, which is why they show up in every hypothesis test.

| Distribution | Arises from | Used in |
|---|---|---|
| **Student's t** | Sample mean when $\sigma$ is unknown and estimated by $s$ | t-tests, CIs (Notebook 7) |
| **Chi-square** $\chi^2$ | Sum of squared standard Normals | Goodness-of-fit, independence (Notebook 8) |
| **F** | Ratio of two chi-squares | ANOVA, comparing variances |

**Student's t** looks like a Normal but with heavier tails, because estimating $\sigma$
adds extra uncertainty. As the degrees of freedom grow, it converges to the Normal — by
$df \approx 30$ the two are nearly indistinguishable.

In [ ]:
xs = np.linspace(-5, 5, 600)
plt.plot(xs, stats.norm.pdf(xs), "k-", lw=2, label="Normal(0,1)")
for df in (1, 3, 10, 30):
    plt.plot(xs, stats.t(df).pdf(xs), lw=1.5, ls="--", label=f"t(df={df})")
plt.legend(); plt.title("Student's t has heavier tails; it converges to Normal as df grows")
plt.xlabel("x"); plt.ylabel("density")
plt.show()

print("Critical value for a two-sided 95% interval:")
for df in (5, 10, 30, 100, 1000):
    print(f"  df={df:>5}: t = {stats.t(df).ppf(0.975):.3f}")
print(f"  Normal  : z = {stats.norm.ppf(0.975):.3f}")

In [ ]:
fig, ax = plt.subplots(1, 2, figsize=(12, 4))

xs = np.linspace(0.01, 30, 500)
for df in (1, 3, 5, 10):
    ax[0].plot(xs, stats.chi2(df).pdf(xs), lw=2, label=f"df={df}")
ax[0].set_ylim(0, 0.5); ax[0].legend(); ax[0].set_title("Chi-square: sum of df squared Normals")

xs = np.linspace(0.01, 5, 500)
for d1, d2 in [(5, 10), (10, 10), (20, 40)]:
    ax[1].plot(xs, stats.f(d1, d2).pdf(xs), lw=2, label=f"F({d1},{d2})")
ax[1].legend(); ax[1].set_title("F: ratio of two scaled chi-squares")
plt.tight_layout(); plt.show()

# Demonstrate the definition of chi-square
z = rng.normal(size=(100_000, 5))
chi_sq_sim = (z ** 2).sum(axis=1)
print(f"Simulated mean of sum of 5 squared Normals: {chi_sq_sim.mean():.3f}  (chi2 df=5 mean = 5)")

---
## 3.10 Choosing the right distribution

Work through these questions in order:

```
Is the outcome a COUNT or a MEASUREMENT?

COUNT ─┬─ Exactly two outcomes, one trial? ............ Bernoulli
       ├─ Successes in a FIXED number of trials? ....... Binomial
       ├─ Events in a fixed time/space window? ......... Poisson
       └─ Trials until the first success? .............. Geometric

MEASUREMENT ─┬─ Symmetric, sum of many small effects? .. Normal
             ├─ Positive, right-skewed, multiplicative?  Log-normal
             ├─ Waiting time, constant hazard rate? .... Exponential
             ├─ No information beyond a range? ......... Uniform
             └─ A statistic computed from a sample? .... t / chi-square / F
```

**Sanity checks before you commit:**
1. Does the support match? (Can your data be negative? Non-integer?)
2. Does the mean/variance relationship match? (Poisson requires mean ≈ variance)
3. Plot a histogram against the fitted PDF
4. Draw a **Q–Q plot** — the most sensitive visual check

In [ ]:
# Fitting a distribution to data with scipy's .fit()
true_data = rng.normal(loc=63, scale=9, size=800)

mu_hat, sigma_hat = stats.norm.fit(true_data)
print(f"Fitted Normal : mu = {mu_hat:.2f}, sigma = {sigma_hat:.2f}   (truth: 63, 9)")

xs = np.linspace(true_data.min(), true_data.max(), 300)
plt.hist(true_data, bins=35, density=True, alpha=0.55, color="steelblue", edgecolor="white")
plt.plot(xs, stats.norm(mu_hat, sigma_hat).pdf(xs), lw=2, color="crimson", label="fitted Normal")
plt.legend(); plt.title("Maximum-likelihood fit")
plt.show()

### Q–Q plots: the fit check that actually works

A **quantile–quantile plot** puts your sorted data on one axis and the quantiles the
theoretical distribution predicts on the other. If the distribution is right, the points
fall on a straight line. Deviations at the ends reveal tail problems that a histogram
hides.

In [ ]:
skewed = rng.lognormal(0, 0.7, 500)
symmetric = rng.normal(0, 1, 500)

fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
stats.probplot(symmetric, dist="norm", plot=ax[0])
ax[0].set_title("Normal data vs Normal quantiles: straight line = good fit")

stats.probplot(skewed, dist="norm", plot=ax[1])
ax[1].set_title("Log-normal data vs Normal quantiles: curved = bad fit")
plt.tight_layout(); plt.show()

In [ ]:
# Formal goodness-of-fit tests
for name, sample in [("Normal sample", symmetric), ("Log-normal sample", skewed)]:
    stat, p = stats.shapiro(sample)
    print(f"{name:20s} Shapiro-Wilk W = {stat:.4f}, p = {p:.2e} "
          f"-> {'consistent with Normal' if p > 0.05 else 'NOT Normal'}")

print()
print("Caution: with large n, normality tests reject almost any real dataset because")
print("no real data is exactly Normal. Always look at the Q-Q plot as well.")

---
## 3.11 Preview: the Central Limit Theorem

Why does the Normal distribution dominate statistics? Because of this:

> **CLT.** For independent, identically distributed $X_i$ with mean $\mu$ and finite
> variance $\sigma^2$, the sample mean $\bar{X}_n$ is approximately
> $N\!\left(\mu, \dfrac{\sigma^2}{n}\right)$ for large $n$ — **no matter what the
> original distribution looks like.**

Watch a strongly skewed exponential population produce beautifully Normal sample means.
Notebook 4 develops this properly.

In [ ]:
population = rng.exponential(scale=3, size=1_000_000)   # very skewed

fig, axes = plt.subplots(1, 4, figsize=(16, 3.4))
axes[0].hist(population[:20_000], bins=60, color="darkorange", edgecolor="none")
axes[0].set_title("Population (Exponential)\nvery skewed")

for ax, n in zip(axes[1:], [2, 10, 50]):
    means = rng.choice(population, size=(20_000, n)).mean(axis=1)
    ax.hist(means, bins=60, color="steelblue", edgecolor="none")
    ax.set_title(f"Means of samples of n={n}\nsd = {means.std():.3f}")
plt.tight_layout(); plt.show()

print(f"Population mean {population.mean():.3f}, sd {population.std():.3f}")
print(f"Predicted sd of the mean at n=50: {population.std()/np.sqrt(50):.3f}")

---
## Exercises

**Exercise 1.** A factory produces items with a 3% defect rate. A quality inspector samples
80 items.
(a) What distribution applies, and why?
(b) $P(\text{no defects})$?  (c) $P(\text{at most 2 defects})$?
(d) How many defects would be surprising (top 1%)?

In [ ]:
# --- Solution 1 -------------------------------------------------------------
# (a) Binomial: fixed n = 80, binary outcome, same p, independent items.
n, p = 80, 0.03
X = stats.binom(n, p)

print(f"(b) P(0 defects)       = {X.pmf(0):.4f}")
print(f"(c) P(at most 2)       = {X.cdf(2):.4f}")
print(f"(d) 99th percentile    = {X.ppf(0.99):.0f} defects")
print(f"    P(X >= 7)          = {X.sf(6):.4f}  -> 7+ defects would be strong evidence")
print(f"\nExpected {X.mean():.1f} defects (sd {X.std():.2f})")

**Exercise 2.** A call centre receives 12 calls per hour on average.
(a) $P(\text{exactly 15 calls in an hour})$
(b) $P(\text{more than 5 calls in 20 minutes})$
(c) What is the distribution of the *time between* calls, and what is its mean in minutes?
(d) $P(\text{a gap longer than 10 minutes})$

In [ ]:
# --- Solution 2 -------------------------------------------------------------
hourly = stats.poisson(12)
print(f"(a) P(exactly 15 in an hour) = {hourly.pmf(15):.4f}")

twenty_min = stats.poisson(12 / 3)            # lambda scales with the window
print(f"(b) P(more than 5 in 20 min) = {twenty_min.sf(5):.4f}")

gap = stats.expon(scale=60 / 12)              # mean 5 minutes between calls
print(f"(c) Gaps are Exponential with mean {gap.mean():.1f} minutes")
print(f"(d) P(gap > 10 min)          = {gap.sf(10):.4f}")

**Exercise 3.** Exam marks are $N(68, 12^2)$.
(a) What fraction of students score above 85?
(b) What mark is the 90th percentile?
(c) The top 5% get a distinction — what is the cut-off?
(d) If 30 students are picked at random, what is the distribution of their *average* mark?

In [ ]:
# --- Solution 3 -------------------------------------------------------------
M = stats.norm(68, 12)
print(f"(a) P(mark > 85)     = {M.sf(85):.4f}  ({M.sf(85)*100:.1f}% of students)")
print(f"(b) 90th percentile  = {M.ppf(0.90):.1f}")
print(f"(c) Distinction cut  = {M.ppf(0.95):.1f}")

# (d) By the CLT (exact here, since the population is Normal):
n = 30
se = 12 / np.sqrt(n)
print(f"(d) Mean of 30 ~ N(68, {se:.3f}^2)  -- the standard error is {se:.3f}")
print(f"    P(class average > 72) = {stats.norm(68, se).sf(72):.4f}")
print("    Note it is far harder for an AVERAGE to be extreme than an individual:")
print(f"    P(individual > 72)    = {M.sf(72):.4f}")

**Exercise 4 (challenge).** You are given an unlabelled dataset. Decide which distribution
generated it by comparing candidate fits with a Q–Q plot and the Kolmogorov–Smirnov test.

In [ ]:
# --- Solution 4 -------------------------------------------------------------
mystery = rng.gamma(shape=2.0, scale=3.0, size=1_000)      # pretend we don't know this

candidates = {
    "norm":      stats.norm,
    "expon":     stats.expon,
    "lognorm":   stats.lognorm,
    "gamma":     stats.gamma,
}

results = []
for name, dist in candidates.items():
    params = dist.fit(mystery)
    ks_stat, p = stats.kstest(mystery, name, args=params)
    results.append({"distribution": name, "KS statistic": ks_stat, "p-value": p})

res = pd.DataFrame(results).sort_values("KS statistic")
print(res.to_string(index=False))
print("\nSmallest KS statistic (and a p-value above 0.05) = best fit.")

best = res.iloc[0]["distribution"]
fig, ax = plt.subplots(1, 2, figsize=(12, 4.2))
stats.probplot(mystery, dist=candidates[best], sparams=candidates[best].fit(mystery), plot=ax[0])
ax[0].set_title(f"Q-Q against best fit: {best}")
stats.probplot(mystery, dist="norm", plot=ax[1])
ax[1].set_title("Q-Q against Normal (for comparison)")
plt.tight_layout(); plt.show()

---
## Summary

| Distribution | Parameters | Mean | Variance | Typical use |
|---|---|---|---|---|
| Bernoulli | $p$ | $p$ | $p(1-p)$ | One yes/no trial |
| Binomial | $n, p$ | $np$ | $np(1-p)$ | Successes in $n$ trials |
| Poisson | $\lambda$ | $\lambda$ | $\lambda$ | Counts per window |
| Geometric | $p$ | $1/p$ | $(1-p)/p^2$ | Trials until first success |
| Uniform | $a, b$ | $(a+b)/2$ | $(b-a)^2/12$ | Equal likelihood over a range |
| Normal | $\mu, \sigma$ | $\mu$ | $\sigma^2$ | Sums of many small effects |
| Exponential | $\lambda$ | $1/\lambda$ | $1/\lambda^2$ | Waiting times |
| Log-normal | $\mu, \sigma$ | $e^{\mu+\sigma^2/2}$ | — | Multiplicative, positive, skewed |
| $t$ | $df$ | 0 | $df/(df-2)$ | Means with unknown $\sigma$ |
| $\chi^2$ | $df$ | $df$ | $2\,df$ | Variances, categorical counts |

**Next up:** [Notebook 4 — Sampling Techniques](4.%20Sampling%20Techniques.ipynb) — how to
get a sample that actually represents the population, and what the Central Limit Theorem
guarantees about it.